0. Environment Configuration

0.1 Install dependencies according to requirement.txt

0.2 Make sure your directory structure like this:

your_project/

├── models.py # Your provided model files

├── modules.py # CellEncoder and other modules

├── config_files/

│ └── config_past_train.py # Configuration files

├── utils/

│ ├── data.py # Data processing functions

│ ├── util.py # Utility functions

│ └── constants.py # Constant definitions

└── extract_embeddings.py # Extraction script (see below)

0.3 Required Documents
You will need to prepare the following files (download them by hugging face):

Model weight file, that is Trained checkpoint file checkpoint_context_1500.pth: https://huggingface.co/Boyoungc/PAST_SC_Embedding/blob/main/checkpoint_context_1500.pth

Gene reference file, model.h5ad (Contains a list of genes used during model training): https://huggingface.co/Boyoungc/PAST_SC_Embedding/blob/main/model.h5ad

Gene mean file, xenium_mean_script.npy (Gene mean vector required for tokenization): https://huggingface.co/Boyoungc/PAST_SC_Embedding/blob/main/xenium_mean_script.npy


1. Data preparation

1.1 Single-cell data format requirements

You need to prepare single-cell data in AnnData format:

In [1]:
import scanpy as sc

# load your single cell data
adata = sc.read_h5ad("single_cell_data.h5ad")

# Required content:
# - adata.X:  [n_cells × n_genes]
# - adata.var_names: gene names
# - adata.obs: Cell metadata (optional)

print(f"cells: {adata.n_obs}")
print(f"genes: {adata.n_vars}")

/home/yangc0h/miniconda3/envs/clip_dl/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/yangc0h/miniconda3/envs/clip_dl/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/yangc0h/miniconda3/envs/clip_dl/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/yangc0h/miniconda3/envs/clip_dl/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/yangc0h/miniconda3/envs/clip_dl/lib/python3.11/site-packages/anndata/utils.py:429: Futur

cells: 200
genes: 10


/home/yangc0h/miniconda3/envs/clip_dl/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_umi_tools from `anndata` is deprecated. Import anndata.io.read_umi_tools instead.
  warnings.warn(msg, FutureWarning)


1.2 Adding Required Metadata

To ensure compatibility with our model, the following metadata needs to be added:

In [2]:
import pandas as pd

# Add species information
adata.obs['organism'] = 'human' # or 'mouse'

# Add technology platform information
adata.obs['assay'] = "10x 3' v3" # Choose according to actual situation

# Add modality information
adata.obs['modality'] = 'dissociated'

# Complete list of technology platforms (refer to your technology_dict):

# "10x 5' v2", "10x 3' v3", "10x 3' v2", "10x 5' v1",

# "10x 3' v1", "CITE-seq", "Smart-seq v4", etc.

# here is I used for pretraining:
modality_dict = {
    'dissociated': 3,
    'spatial': 4,}

specie_dict = {
    'human': 5,
    'Homo sapiens': 5,
    'Mus musculus': 6,
    'mouse': 6,}

technology_dict = {
    "merfish": 7,
    "MERFISH": 7,
    "cosmx": 8,
    "NanoString digital spatial profiling": 8,
    "Xenium": 9,
    "10x 5' v2": 10,
    "10x 3' v3": 11,
    "10x 3' v2": 12,
    "10x 5' v1": 13,
    "10x 3' v1": 14,
    "10x 3' transcription profiling": 15, 
    "10x transcription profiling": 15,
    "10x 5' transcription profiling": 16,
    "CITE-seq": 17, 
    "Smart-seq v4": 18,
}


2. Extract Embedding

2.1 Gene alignment

In [3]:
import scanpy as sc
import numpy as np
from utils.data import align_genes_with_model
import scipy

# 1. Load model gene reference
model = sc.read_h5ad("/ibex/project/c2277/data/Pathomics/preprocessed/model_means/model.h5ad") # Note: Please replace the file paths in this example with the locations 
# where you downloaded the files. Also note that the provided model uses Ensembl Gene IDs as gene names (e.g., ENSG00000000003). 
# If your dataset uses a different gene naming convention (e.g., HGNC gene symbols), you must convert your gene identifiers to Ensembl IDs before running the code. 
# This is straightforward and can be done using mygene (recommended) or gprofiler-official.

print(model.var_names)
print(f"Number of genes in the model: {len(model.var_names)}")

# 2. Load your single-cell data
adata = sc.read_h5ad("single_cell_data.h5ad")
print(f"Number of genes in the original model: {adata.n_vars}")

# 3. Gene alignment - only keep genes present in the model
adata_aligned, removed_genes = align_genes_with_model(model, adata)
print(f"Number of genes after alignment: {adata_aligned.n_vars}")
print(f"Number of genes removed: {len(removed_genes)}")

# 4. Process the sparse matrix and fill missing genes with 0

if scipy.sparse.issparse(adata_aligned.X):

    X_aligned = adata_aligned.X.toarray()

else:

    X_aligned = adata_aligned.X

# If the number of aligned genes is less than the number of model genes, fill with 0

if adata_aligned.n_vars < len(model.var_names):

    n_cells = X_aligned.shape[0]
    
    n_model_genes = len(model.var_names)
    
    X_full = np.zeros((n_cells, n_model_genes))
    
    # Find the position of aligned genes in the model
    
    aligned_gene_indices = [model.var_names.get_loc(g) for g in adata_aligned.var_names]
    
    X_full[:, aligned_gene_indices] = X_aligned
    
    X_aligned = X_full

Index(['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419',
       'ENSG00000000457', 'ENSG00000000460', 'ENSG00000000938',
       'ENSG00000000971', 'ENSG00000001036', 'ENSG00000001084',
       'ENSG00000001167',
       ...
       'ENSMUSG00000112117', 'ENSMUSG00000112148', 'ENSMUSG00000112276',
       'ENSMUSG00000113186', 'ENSMUSG00000114028', 'ENSMUSG00000114469',
       'ENSMUSG00000115424', 'ENSMUSG00000115432', 'ENSMUSG00000115529',
       'ENSMUSG00000115965'],
      dtype='object', length=20310)
Number of genes in the model: 20310
Number of genes in the original model: 10
Number of genes after alignment: 10
Number of genes removed: 0


2.2 Data Tokenization

In [4]:
from utils.data import tokenize_data

# 1. Load gene mean
ori_model_mean = np.load("/ibex/project/c2277/data/Pathomics/preprocessed/model_means/xenium_mean_script.npy") # Note: Please replace the file paths in this example with the locations 
# where you downloaded the files.

# 2. Preprocess the mean (same as during training)
ori_model_mean = np.nan_to_num(ori_model_mean)
rounded_values = np.where(
    (ori_model_mean % 1) >= 0.5, 
    np.ceil(ori_model_mean), 
    np.floor(ori_model_mean)
)
ori_model_mean = np.where(ori_model_mean == 0, 1, rounded_values)

# 3. Tokenization
tokenized, processed_x = tokenize_data(
    X_aligned, 
    ori_model_mean, 
    max_seq_len=4096  # 与训练时一致
)

print(f"Tokenized shape: {tokenized.shape}")  # [n_cells, 4096]

Tokenized shape: (200, 4096)


2.3 Preparing Model Input

In [5]:
from utils.util import preprocess_cell_data, complete_masking
import config_files.config_past_train as CFGP
import torch

# Encoding of technology platforms and species (refer to my tokenize script)
modality_dict = {
    'dissociated': 3,
    'spatial': 4,}

specie_dict = {
    'human': 5,
    'Homo sapiens': 5,
    'Mus musculus': 6,
    'mouse': 6,}

technology_dict = {
    "merfish": 7,
    "MERFISH": 7,
    "cosmx": 8,
    "NanoString digital spatial profiling": 8,
    "Xenium": 9,
    "10x 5' v2": 10,
    "10x 3' v3": 11,
    "10x 3' v2": 12,
    "10x 5' v1": 13,
    "10x 3' v1": 14,
    "10x 3' transcription profiling": 15, 
    "10x transcription profiling": 15,
    "10x 5' transcription profiling": 16,
    "CITE-seq": 17, 
    "Smart-seq v4": 18,
}

# batch data
batch = {
    'X': torch.tensor(tokenized, dtype=torch.long),
    'modality': torch.tensor([modality_dict['dissociated']] * len(tokenized)),
    'assay': torch.tensor([technology_dict["10x 3' v3"]] * len(tokenized)),
    'specie': torch.tensor([specie_dict['human']] * len(tokenized)),
}

# preprocessing
processed_batch = preprocess_cell_data(batch, CFGP)

# If masking was used during training, it is set to 0 here (no masking during inference).
masked_batch = complete_masking(
    processed_batch, 
    p=0.0,  
    n_tokens=CFGP.cell_n_tokens + 5
)

2.4 Load the model and extract the embedding

In [8]:
import torch
from models import PASTModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Initialize the model
model = PASTModel()

# 2. Load Weights
checkpoint = torch.load('/ibex/project/c2277/data/Pathomics/preprocessed/model_means/checkpoint_context_1500.pth', map_location=device) # Note: Please replace the file paths in this example with the locations 
# where you downloaded the files.

# Process the model trained using DDP (remove the 'module.' prefix)
state_dict = checkpoint.get('model', checkpoint)
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith('module.'):
        new_state_dict[k[7:]] = v
    else:
        new_state_dict[k] = v

model.load_state_dict(new_state_dict, strict=False)
model.to(device)
model.eval()

# 3. Batch extract embeddings
batch_size = 8
all_embeddings = []

with torch.no_grad():
    for i in range(0, len(tokenized), batch_size):
        # prepare batch
        batch_indices = slice(i, min(i + batch_size, len(tokenized)))
        
        batch_tokens = masked_batch['masked_indices'][batch_indices].to(device)
        batch_attention = masked_batch['attention_mask'][batch_indices].to(device)
        
        # Extract embedding
        embeddings = model.cell_encoder(batch_tokens, batch_attention)
        
        all_embeddings.append(embeddings.cpu().numpy())

# 4. Merge all embeddings
final_embeddings = np.vstack(all_embeddings)
print(f"final embedding shape: {final_embeddings.shape}")  # [n_cells, 512]

Unexpected keys in pretrained weights: ['classifier_head.weight', 'classifier_head.bias', 'pooler_head.weight', 'pooler_head.bias', 'cls_head.weight', 'cls_head.bias']
final embedding shape: (200, 512)
